# stainkit — GPU-accelerated H&E stain normalization

This notebook clones the stainkit repository, builds it, downloads a
sample dataset, runs the pipeline, and pushes the results back to the
repo. It expects a `GITHUB_TOKEN` secret in this notebook's settings.

In [ ]:
from google.colab import drive, userdata
import subprocess, os, sys, pathlib

# ---- 1. authenticate ----
token = userdata.get('GITHUB_TOKEN')
if not token or not token.startswith('ghp_'):
    raise SystemExit('GITHUB_TOKEN is missing or invalid.')
GITHUB_USER  = '404Piyush'
REPO_NAME    = 'stainkit'
REPO_URL     = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
print(f'Using {GITHUB_USER}/{REPO_NAME}')

In [ ]:
# ---- 2. clone ----
if pathlib.Path(REPO_NAME).exists():
    print('Repository already present, pulling latest changes.')
    subprocess.run(['git', '-C', REPO_NAME, 'pull', '--rebase', '--autostash'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_NAME], check=True)
%cd {REPO_NAME}
subprocess.run(['git', 'config', 'user.email', 'colab@stainkit.local'], check=True)
subprocess.run(['git', 'config', 'user.name',  'colab-runner'],        check=True)

In [ ]:
# ---- 3. install system + python packages ----
!bash scripts/setup_colab.sh

In [ ]:
# ---- 4. fetch stb_image (stb repo ships on master branch only) ----
!mkdir -p third_party/stb && cd third_party/stb && \
  curl -fsSL -o stb_image.h       https://raw.githubusercontent.com/nothings/stb/master/stb_image.h && \
  curl -fsSL -o stb_image_write.h https://raw.githubusercontent.com/nothings/stb/master/stb_image_write.h && \
  cd ../..

In [ ]:
# ---- 5. build ----
!./install.sh

In [ ]:
# ---- 6. download a sample dataset ----
!python scripts/download_pcam.py --num-images 100 --output data/raw

In [ ]:
# ---- 7. run the pipeline ----
!./run.sh --num 100 --target default --stream 4

In [ ]:
# ---- 8. plot the benchmark ----
!python scripts/benchmark.py data/benchmark/last_run.csv \
  --output data/benchmark/last_run.png

In [ ]:
# ---- 9. inspect a few output images ----
from IPython.display import Image, display
import os, random
files = sorted(f for f in os.listdir('data/processed') if f.endswith('_panel.png'))
for f in random.sample(files, k=min(3, len(files))):
    display(Image(f'data/processed/{f}'))

In [ ]:
# ---- 10. push results back to the repo ----
subprocess.run(['git', 'add', 'data/processed', 'data/benchmark', 'docs/benchmarks.md'], check=True)
subprocess.run(['git', 'commit', '-m', 'colab: run results'], check=True)
subprocess.run(['git', 'push', 'origin', 'main'], check=True)
print('Pushed.')